# FMCW training
Train the single-chirp model with the supplied datasets and fixed splits.
This notebook imports the code in `fmcw_training/`; it does not duplicate it.
For Colab, put this complete folder in `MyDrive/fmcw-mitigation-detection/`.
The supplied epoch-115 weights are separate from the new run created below.


## 1. Project and dependencies


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
try:
    from google.colab import drive
except ImportError:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT = next((p for p in candidates if (p/'training/fmcw_training').is_dir()), Path.cwd())
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/fmcw-mitigation-detection')
# Change PROJECT above if your folder is elsewhere.
assert (PROJECT/'training/fmcw_training').is_dir(), 'Set PROJECT to the repository folder.'
sys.path.insert(0, str(PROJECT/'training'))
missing = [name for name in ['torch', 'numpy', 'h5py', 'matplotlib', 'tqdm']
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r',
                           str(PROJECT/'training/requirements.txt')])
print('Project:', PROJECT)


## 2. Settings
`EPOCHS` is the total training limit. Reuse a compatible `RUN_DIR` to resume
a run at the last complete epoch. Use a new directory for a new experiment.
Training and test evaluation are disabled until their flags are enabled.


In [ ]:
SEED = 12
EPOCHS = 120
BATCH_SIZE = 16
EFFECTIVE_BATCH = 32
LEARNING_RATE = 3e-4
RUN_DIR = PROJECT/'training/runs/seed12'
RUN_TRAINING = False
RUN_EVALUATION = False


## 3. Load datasets and splits
The external catalog is read for metadata checks. No external waveform
inference occurs here. The default seed uses the supplied exact split.


In [ ]:
import json
import tempfile
from fmcw_training.data import Catalog, materialize, split_catalog
cache = Path(tempfile.gettempdir())/'fmcw_dataset_cache'
main = Catalog(materialize(PROJECT/'dataset/principal.zip', cache/'principal'))
external = Catalog(materialize(PROJECT/'dataset/externe.zip', cache/'externe'))
main.independent_of(external)
splits = split_catalog(main, SEED)
provided = json.loads((PROJECT/'dataset/partitions.json').read_text())
if SEED == provided['seed']:
    assert splits == provided['splits'], 'The datasets do not match the supplied split.'
print({name: len(indices) for name, indices in splits.items()})


## 4. Train or resume
The model learns a complex reconstruction and a presence profile. Labels come
from target positions; echo amplitude does not set their peak height. The input
uses only the observed waveform and known radar settings. Curves use threshold 0.5.


In [ ]:
from fmcw_training.model import ModelConfig
from fmcw_training.engine import Experiment, TrainConfig
model_config = ModelConfig(width=24, depth=4, modes=24, conditioning=True,
                           spectral=True, temporal=True, stft=True)
train_config = TrainConfig(seed=SEED, epochs=EPOCHS, batch_size=BATCH_SIZE,
                           effective_batch=EFFECTIVE_BATCH, lr=LEARNING_RATE,
                           patience=20, fp_budget=0.1, workers=0, plot_every=5)
experiment = Experiment(RUN_DIR/'complet', main, splits, external,
                        model_config=model_config, train_config=train_config)
if RUN_TRAINING:
    experiment.fit(show=True)
else:
    print('Training disabled. Set RUN_TRAINING=True when ready.')


## 5. Saved learning curves


In [ ]:
from IPython.display import Image, display
curves = RUN_DIR/'complet/courbes.png'
if curves.exists():
    display(Image(filename=str(curves)))
else:
    print('No curves yet for this run. Supplied curves: training/learning_curves.png')


## 6. Optional evaluation and inference export
Keep this disabled during model selection. Calibration uses validation only.
The supplied external scenes have already been evaluated during development;
reusing them is a secondary evaluation. Use new scenes for independent confirmation.
The supplied epoch-115 inference weights have completed this secondary evaluation.
The cell below evaluates the separate run configured in this notebook.

Checkpoint selection uses validation F1 at threshold 0.5 before calibration.


In [ ]:
from checkpoint_selection import evaluate_for_detection
if RUN_EVALUATION:
    results = evaluate_for_detection(experiment, show=True)
    for name in ['test_interne', 'validation_externe']:
        print(name, results[name]['presence'])
    print('Inference export:', RUN_DIR/'complet/inference.pt')
else:
    print('Test evaluation disabled.')
